# Hour 1 — What is a window function?

In [1]:
import pandas as pd
import duckdb

In [2]:
sales = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "customer": ["Alex", "Maria", "James", "Sarah", "Alex", "Maria", "James", "Alex"],
    "state": ["CA", "CA", "TX", "NY", "CA", "CA", "TX", "CA"],
    "total_sales": [600, 1200, 300, 300, 1200, 200, 600, 400]
})

display(sales)

,order_id,customer,state,total_sales
0,1001,Alex,CA,600
1,1002,Maria,CA,1200
2,1003,James,TX,300
3,1004,Sarah,NY,300
4,1005,Alex,CA,1200
5,1006,Maria,CA,200
6,1007,James,TX,600
7,1008,Alex,CA,400


ROW_NUMBER() OVER()

In [5]:
query = """
SELECT 
order_id, 
customer, 
total_sales,
ROW_NUMBER() OVER(
ORDER BY total_sales DESC)
AS sales_order
FROM sales;"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,sales_order
0,1002,Maria,1200,1
1,1005,Alex,1200,2
2,1001,Alex,600,3
3,1007,James,600,4
4,1008,Alex,400,5
5,1003,James,300,6
6,1004,Sarah,300,7
7,1006,Maria,200,8


PARTITION BY

In [6]:
query = """
SELECT 
order_id, 
customer, 
total_sales,
ROW_NUMBER() OVER(
PARTITION BY customer
ORDER BY total_sales DESC)
AS sales_order
FROM sales;"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,sales_order
0,1004,Sarah,300,1
1,1007,James,600,1
2,1003,James,300,2
3,1005,Alex,1200,1
4,1001,Alex,600,2
5,1008,Alex,400,3
6,1002,Maria,1200,1
7,1006,Maria,200,2


Change the partition

In [7]:
query = """
SELECT
order_id,
customer,
state,
total_sales,
ROW_NUMBER() OVER(
PARTITION BY state
ORDER BY total_sales DESC)
AS state_sales_order
FROM sales;
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,state,total_sales,state_sales_order
0,1002,Maria,CA,1200,1
1,1005,Alex,CA,1200,2
2,1001,Alex,CA,600,3
3,1008,Alex,CA,400,4
4,1006,Maria,CA,200,5
5,1007,James,TX,600,1
6,1003,James,TX,300,2
7,1004,Sarah,NY,300,1


Combine CTE + Window Function

In [11]:
query = """
WITH ranked_sales AS (
SELECT
customer,
order_id,
total_sales,
ROW_NUMBER() OVER(
PARTITION BY customer
ORDER BY total_sales DESC)
AS sales_order 
FROM sales)

SELECT
customer,
order_id,
total_sales
FROM ranked_sales
WHERE sales_order = 1"""

result = duckdb.sql(query).df()
display(result)

,customer,order_id,total_sales
0,James,1007,600
1,Sarah,1004,300
2,Alex,1005,1200
3,Maria,1002,1200


In [12]:
query = """
WITH ranked_sales AS (
SELECT
state,
customer,
order_id,
total_sales,
ROW_NUMBER() OVER(
PARTITION BY state
ORDER BY total_sales DESC)
AS state_rank
FROM sales)

SELECT
state,
customer,
order_id,
total_sales,
state_rank
FROM ranked_sales
WHERE state_rank <=2
"""

result = duckdb.sql(query).df()
display(result)

,state,customer,order_id,total_sales,state_rank
0,NY,Sarah,1004,300,1
1,CA,Alex,1005,1200,1
2,CA,Maria,1002,1200,2
3,TX,James,1007,600,1
4,TX,James,1003,300,2


Hour 2: ROW_NUMBER() vs RANK() vs DENSE_RANK()

In [7]:
query = """
SELECT
order_id,
customer,
total_sales,
RANK() OVER (
ORDER BY total_sales DESC
) AS sales_rank
FROM sales;
"""

result = duckdb.sql(query).df()

display(result)

,order_id,customer,total_sales,sales_rank
0,1002,Maria,1200,1
1,1005,Alex,1200,1
2,1001,Alex,600,3
3,1007,James,600,3
4,1008,Alex,400,5
5,1003,James,300,6
6,1004,Sarah,300,6
7,1006,Maria,200,8


In [9]:
query = """
SELECT
    order_id,
    customer,
    total_sales,
    DENSE_RANK() OVER(
        ORDER BY total_sales DESC)
    AS sales_rank
FROM sales;"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,sales_rank
0,1002,Maria,1200,1
1,1005,Alex,1200,1
2,1001,Alex,600,2
3,1007,James,600,2
4,1008,Alex,400,3
5,1003,James,300,4
6,1004,Sarah,300,4
7,1006,Maria,200,5


Combine ranking with PARTITION BY

In [10]:
query = """
SELECT
    state,
    customer,
    order_id,
    total_sales,
    DENSE_RANK() OVER(
        PARTITION BY state
        ORDER BY total_sales DESC)
    AS state_rank
FROM sales;  """

result = duckdb.sql(query).df()
display(result)

,state,customer,order_id,total_sales,state_rank
0,CA,Maria,1002,1200,1
1,CA,Alex,1005,1200,1
2,CA,Alex,1001,600,2
3,CA,Alex,1008,400,3
4,CA,Maria,1006,200,4
5,TX,James,1007,600,1
6,TX,James,1003,300,2
7,NY,Sarah,1004,300,1


Find the top 2 distinct sales amounts in each state. If multiple orders tie at either sales amount, include all of them.

In [11]:
query = """
WITH state_sales AS (
SELECT
    state,
    customer,
    order_id,
    total_sales,
    DENSE_RANK() OVER(
        PARTITION BY state
        ORDER BY total_sales DESC)
    AS state_rank
FROM sales)

SELECT 
    state,
    customer,
    order_id,
    total_sales,
    state_rank
FROM state_sales
WHERE state_rank <= 2 """

result = duckdb.sql(query).df()
display(result)

,state,customer,order_id,total_sales,state_rank
0,NY,Sarah,1004,300,1
1,CA,Maria,1002,1200,1
2,CA,Alex,1005,1200,1
3,CA,Alex,1001,600,2
4,TX,James,1007,600,1
5,TX,James,1003,300,2


Hour 3: Practical Window Analytics

Running totals

In [4]:
query = """
SELECT
order_id,
customer,
total_sales,
SUM(total_sales) OVER(
    ORDER BY order_id)
AS running_sales
FROM sales;"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,running_sales
0,1001,Alex,600,600.0
1,1002,Maria,1200,1800.0
2,1003,James,300,2100.0
3,1004,Sarah,300,2400.0
4,1005,Alex,1200,3600.0
5,1006,Maria,200,3800.0
6,1007,James,600,4400.0
7,1008,Alex,400,4800.0


Running total per customer

In [6]:
query = """
SELECT
    order_id,
    customer,
    total_sales,
    SUM(total_sales) OVER(
        PARTITION BY customer
        ORDER BY order_id)
    AS customer_running_sales
FROM sales;"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,customer_running_sales
0,1004,Sarah,300,300.0
1,1001,Alex,600,600.0
2,1005,Alex,1200,1800.0
3,1008,Alex,400,2200.0
4,1002,Maria,1200,1200.0
5,1006,Maria,200,1400.0
6,1003,James,300,300.0
7,1007,James,600,900.0


LAG()

In [7]:
query = """
SELECT
    order_id,
    customer,
    total_sales,
    LAG(total_sales) OVER(
        PARTITION BY customer
        ORDER BY order_id)
    AS previous_sale
FROM sales; """

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,previous_sale
0,1002,Maria,1200,<NA>
1,1006,Maria,200,1200
2,1004,Sarah,300,<NA>
3,1003,James,300,<NA>
4,1007,James,600,300
5,1001,Alex,600,<NA>
6,1005,Alex,1200,600
7,1008,Alex,400,1200


In [16]:
query = """
WITH order_difference AS (
SELECT
    order_id,
    customer,
    total_sales,
    LAG(total_sales) OVER(
        PARTITION BY customer
        ORDER BY order_id)
    AS previous_sale
FROM sales
)

SELECT
    order_id,
    customer,
    total_sales,
    previous_sale,
    total_sales - previous_sale AS sales_change,
    CASE
        WHEN total_sales - previous_sale > 0 THEN 'Increase'
        WHEN total_sales - previous_sale < 0 THEN 'Decrease'
        WHEN total_sales - previous_sale = 0 THEN 'No Change'
        END AS change_type
FROM order_difference
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,previous_sale,sales_change,change_type
0,1004,Sarah,300,<NA>,<NA>,NaN
1,1001,Alex,600,<NA>,<NA>,NaN
2,1005,Alex,1200,600,600,Increase
3,1008,Alex,400,1200,-800,Decrease
4,1003,James,300,<NA>,<NA>,NaN
5,1007,James,600,300,300,Increase
6,1002,Maria,1200,<NA>,<NA>,NaN
7,1006,Maria,200,1200,-1000,Decrease


For each customer, show each order, the previous order amount, and whether the order increased, decreased, or stayed the same compared with the previous order.

In [17]:
query = """
WITH order_difference AS (
SELECT
    order_id,
    customer,
    total_sales,
    LAG(total_sales) OVER(
        PARTITION BY customer
        ORDER BY order_id)
    AS previous_sale
FROM sales
)

SELECT
    order_id,
    customer,
    total_sales,
    previous_sale,
    total_sales - previous_sale AS sales_change,
    CASE
        WHEN total_sales - previous_sale > 0 THEN 'Increase'
        WHEN total_sales - previous_sale < 0 THEN 'Decrease'
        WHEN total_sales - previous_sale = 0 THEN 'No Change'
        END AS change_type
FROM order_difference
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,total_sales,previous_sale,sales_change,change_type
0,1004,Sarah,300,<NA>,<NA>,NaN
1,1002,Maria,1200,<NA>,<NA>,NaN
2,1006,Maria,200,1200,-1000,Decrease
3,1001,Alex,600,<NA>,<NA>,NaN
4,1005,Alex,1200,600,600,Increase
5,1008,Alex,400,1200,-800,Decrease
6,1003,James,300,<NA>,<NA>,NaN
7,1007,James,600,300,300,Increase


In [18]:
project_sales = pd.read_csv("../data/sales_analysis_clean.csv")
display(project_sales)

,order_id,customer_id,product_id,quantity,customer_name,state,product_name,price,total_sales
0,1001,1,10,2,Alex,CA,Laptop,1200,2400
1,1003,2,10,1,Maria,CA,Laptop,1200,1200
2,1005,3,11,2,James,TX,Monitor,300,600
3,1002,1,11,1,Alex,CA,Monitor,300,300
4,1004,3,12,3,James,TX,Keyboard,100,300


In [19]:
project_sales.columns

Index(['order_id', 'customer_id', 'product_id', 'quantity', 'customer_name',
       'state', 'product_name', 'price', 'total_sales'],
      dtype='str')

For each customer, rank their orders from highest to lowest sales and calculate their cumulative sales over time.

In [22]:
query = """
SELECT
    order_id,
    customer_name,
    total_sales,
    ROW_NUMBER() OVER(
        PARTITION BY customer_name
        ORDER BY total_sales DESC)
    AS sales_rank,
    SUM(total_sales) OVER(
        PARTITION BY customer_name
        ORDER BY order_id)
    AS customer_running_sales
FROM project_sales;
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer_name,total_sales,sales_rank,customer_running_sales
0,1001,Alex,2400,1,2400.0
1,1002,Alex,300,2,2700.0
2,1003,Maria,1200,1,1200.0
3,1004,James,300,2,300.0
4,1005,James,600,1,900.0


Add previous-order analysis

In [24]:
query = """
SELECT
    order_id,
    customer_name,
    total_sales,
    ROW_NUMBER() OVER(
        PARTITION BY customer_name
        ORDER BY total_sales DESC)
    AS sales_rank,
    SUM(total_sales) OVER(
        PARTITION BY customer_name
        ORDER BY order_id)
    AS customer_running_sales,
    LAG(total_sales) OVER(
        PARTITION BY customer_name
        ORDER BY order_id)
    AS previous_sale
FROM project_sales;
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer_name,total_sales,sales_rank,customer_running_sales,previous_sale
0,1001,Alex,2400,1,2400.0,<NA>
1,1002,Alex,300,2,2700.0,2400
2,1004,James,300,2,300.0,<NA>
3,1005,James,600,1,900.0,300
4,1003,Maria,1200,1,1200.0,<NA>


add sales_change

In [26]:
query = """
WITH customer_metrics AS(
    SELECT
        order_id,
        customer_name,
        total_sales,
        ROW_NUMBER() OVER(
            PARTITION BY customer_name
            ORDER BY total_sales DESC)
        AS sales_rank,
        SUM(total_sales) OVER(
            PARTITION BY customer_name
            ORDER BY order_id)
        AS customer_running_sales,
        LAG(total_sales) OVER(
            PARTITION BY customer_name
            ORDER BY order_id)
        AS previous_sale
    FROM project_sales
)

SELECT
    order_id,
    customer_name,
    total_sales,
    sales_rank,
    customer_running_sales,
    previous_sale,
    total_sales - previous_sale AS sales_change
FROM customer_metrics
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer_name,total_sales,sales_rank,customer_running_sales,previous_sale,sales_change
0,1004,James,300,2,300.0,<NA>,<NA>
1,1005,James,600,1,900.0,300,300
2,1001,Alex,2400,1,2400.0,<NA>,<NA>
3,1002,Alex,300,2,2700.0,2400,-2100
4,1003,Maria,1200,1,1200.0,<NA>,<NA>


In [27]:
result.to_csv(
    "../data/day_03_customer_metrics.csv",
    index=False
)

In [29]:
check_day3 = pd.read_csv("../data/day_03_customer_metrics.csv")

check_day3.shape
check_day3.head()

,order_id,customer_name,total_sales,sales_rank,customer_running_sales,previous_sale,sales_change
0,1004,James,300,2,300.0,NaN,NaN
1,1005,James,600,1,900.0,300.0,300.0
2,1001,Alex,2400,1,2400.0,NaN,NaN
3,1002,Alex,300,2,2700.0,2400.0,-2100.0
4,1003,Maria,1200,1,1200.0,NaN,NaN
